In [1]:
import sys

print(sys.version)

# LSTM WAS DONE IN PYTHON 3.10.20

3.10.20 (main, Mar 11 2026, 17:43:48) [Clang 20.1.8 ]


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras import backend as K
import gc

df = pd.read_excel('CLEANED_DATASET_THESIS_FINAL.xlsx') # CHANGE DIRECTORY TO RIGHT PATH

df["TIME_PERIOD"] = pd.to_datetime(df["TIME_PERIOD"].astype(str).str.replace("-M", "-"), format="%Y-%m").dt.to_period("M")

new_rows = []

SEQ_LEN = 12
STEP = 12

def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i2 in range(len(X) - seq_len):
        Xs.append(X[i2:i2+seq_len])
        ys.append(y[i2+seq_len])
    return np.array(Xs), np.array(ys)

for country, group in df.groupby('COUNTRY'):
    
    group = group.sort_values('TIME_PERIOD').copy()

    temp = group[['COUNTRY', 'TIME_PERIOD', 'OBS_VALUE', 'GPR', 'ANNUALIZED_VOLATILITY']].copy()

    temp['OBS_VALUE'] = pd.to_numeric(temp['OBS_VALUE'], errors='coerce')
    temp['GPR'] = pd.to_numeric(temp['GPR'], errors='coerce')

    temp['LAG_3'] = temp['OBS_VALUE'].shift(3)
    temp['ROLL_MEAN_3'] = temp['OBS_VALUE'].rolling(3).mean()
    temp['ROLL_MEAN_12'] = temp['OBS_VALUE'].rolling(12).mean()
    temp['LOG_RETURN'] = np.log(temp['OBS_VALUE'] / temp['OBS_VALUE'].shift(1))

    temp['TARGET'] = temp['OBS_VALUE'].shift(-1)

    temp = temp.dropna().reset_index(drop=True)

    n = len(temp)
    min_train_size = int(n * 0.6)

    all_preds, all_true = [], []
    vol_preds, vol_true = [], []

    indices = list(range(min_train_size, n, STEP))

    for i in indices:

        train = temp.iloc[:i].copy()
        test = temp.iloc[i:i+1].copy()

        # scaling
        scaler_X = MinMaxScaler()
        scaler_y = MinMaxScaler()

        features = [
            'OBS_VALUE',
            'GPR',
            'LAG_3',
            'ROLL_MEAN_3',
            'ROLL_MEAN_12',
            'LOG_RETURN'
        ]

        X_train = train[features].values.astype(np.float32)
        y_train = train[['TARGET']].values.astype(np.float32)

        X_test = test[features].values.astype(np.float32)
        y_test = test[['TARGET']].values.astype(np.float32)

        scaler_X.fit(X_train)
        scaler_y.fit(y_train)

        X_train_scaled = scaler_X.transform(X_train)
        X_test_scaled = scaler_X.transform(X_test)

        y_train_scaled = scaler_y.transform(y_train)
        y_test_scaled = scaler_y.transform(y_test)

        X_train_seq, y_train_seq = create_sequences(
            X_train_scaled, y_train_scaled, SEQ_LEN
        )
        
        model = Sequential([Input(shape=(SEQ_LEN, len(features))), LSTM(8), Dense(1)])

        model.compile(optimizer='adam', loss='mse')

        model.fit(X_train_seq, y_train_seq, epochs=5, batch_size=8,verbose=0)

        last_seq = X_train_seq[-1].reshape(1, SEQ_LEN, len(features))

        y_pred_scaled = model.predict(last_seq, verbose=0)

        y_pred = scaler_y.inverse_transform(y_pred_scaled)[0][0]
        y_true = y_test[0][0]

        all_preds.append(y_pred)
        all_true.append(y_true)

        obs_value = test['OBS_VALUE'].values[0]

        vol_true.append(np.log(y_true / obs_value))
        vol_preds.append(np.log(y_pred / obs_value))

        K.clear_session()
        del model
        gc.collect()

    mae = mean_absolute_error(all_true, all_preds)
    mse = mean_squared_error(all_true, all_preds)

    test_eval = temp.iloc[min_train_size:].copy().iloc[:len(all_preds)]
    test_eval['PRED_OBS'] = all_preds
    test_eval['ACT_OBS'] = all_true

    test_eval['LOG_RET_ACT'] = vol_true
    test_eval['LOG_RET_PRED'] = vol_preds

    test_eval = test_eval.reset_index(drop=True)

    vol_true_arr = np.array(vol_true)
    vol_pred_arr = np.array(vol_preds)
    
    test_eval['VOL_ACT'] = vol_true_arr
    test_eval['VOL_PRED'] = vol_pred_arr

    vol_df = test_eval.dropna(subset=['VOL_ACT', 'VOL_PRED'])

    if len(vol_df) > 1:
        mae_vol = mean_absolute_error(vol_df['VOL_ACT'], vol_df['VOL_PRED'])
        mse_vol = mean_squared_error(vol_df['VOL_ACT'], vol_df['VOL_PRED'])
    else:
        mae_vol = np.nan
        mse_vol = np.nan

    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_full = temp[features].values.astype(np.float32)
    y_full = temp[['TARGET']].values.astype(np.float32)

    scaler_X.fit(X_full)
    scaler_y.fit(y_full)

    X_full_scaled = scaler_X.transform(X_full)
    y_full_scaled = scaler_y.transform(y_full)

    X_seq_full, y_seq_full = create_sequences(
        X_full_scaled, y_full_scaled, SEQ_LEN
    )

    model = Sequential([Input(shape=(SEQ_LEN, len(features))), LSTM(8), Dense(1)])

    model.compile(optimizer='adam', loss='mse')

    model.fit(X_seq_full, y_seq_full, epochs=5, batch_size=8, verbose=0)

    last_sequence = X_seq_full[-1].reshape(1, SEQ_LEN, len(features))

    pred_scaled = model.predict(last_sequence, verbose=0)
    prediction = scaler_y.inverse_transform(pred_scaled)[0][0]

    last_row = temp.iloc[-1]

    new_time = group['TIME_PERIOD'].iloc[-1] + 1

    prev_value = last_row['TARGET']

    per_change = (prediction - prev_value) / prev_value
    log_return = np.log(prediction / prev_value)

    last_11 = temp['LOG_RETURN'].dropna().iloc[-11:]
    last_12 = pd.concat([last_11, pd.Series([log_return])])

    st_dev = last_12.std()
    annualized_vol = st_dev * np.sqrt(12)

    new_rows.append({
        'COUNTRY': country,
        'TIME_PERIOD': new_time,
        'PRED_VAL_LSTM': prediction,
        'PER_CHANGE': per_change,
        'LOG_RETURN': log_return,
        'ST_DEV': st_dev,
        'ANNUALIZED_VOLATILITY': annualized_vol,
        'LSTM_PRED_MAE': mae,
        'LSTM_PRED_MSE': mse,
        'LSTM_VOL_MAE': mae_vol,
        'LSTM_VOL_MSE': mse_vol
    })

    K.clear_session()
    del model
    gc.collect()

df_new = pd.DataFrame(new_rows)
df_updated = pd.concat([df, df_new], ignore_index=True)
df_updated = df_updated.sort_values(['COUNTRY', 'TIME_PERIOD'])
df_updated.to_excel('LSTM_EXPW_Updated_Clean_DF_80-110.xlsx', index=False)

In [ ]:
df_metrics = pd.read_excel('LSTM_EXPW_Updated_Clean_DF.xlsx')

print(df_metrics['LSTM_PRED_MAE'].describe())

print(df_metrics['LSTM_PRED_MSE'].describe())

print(df_metrics['LSTM_VOL_MAE'].describe())

print(df_metrics['LSTM_VOL_MSE'].describe())